<a href="https://colab.research.google.com/github/shawdaena/Parallel-Processing-and-Distributed-System-Lab/blob/main/MatrixMul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Wed Feb  4 06:28:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [ ]:
#matrix multiplication
%%writefile matrix.cu
// %%cu
#include <stdio.h>
#include <stdlib.h>

// Function to initialize a matrix with random values
void initializeMatrix(int *matrix, int rows, int cols) {
    for (int i = 0; i < rows * cols; ++i) {
        matrix[i] = rand() % 10;
    }
}

// Function to print a matrix
void printMatrix(const int *matrix, int rows, int cols) {
    for (int i = 0; i < rows; ++i) {
        for (int j = 0; j < cols; ++j) {
            printf("%d ", matrix[i * cols + j]);
        }
        printf("\n");
    }
}

// CUDA kernel for element-wise matrix multiplication
__global__ void matrixMultiply(int *a, int *b, int *c, int n, int m, int p) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < n && col < p) {
        int sum = 0;
        for (int i = 0; i < m; ++i) {
            sum += a[blockIdx.z * n * m + row * m + i] * b[blockIdx.z * m * p + i * p + col];
        }
        c[blockIdx.z * n * p + row * p + col] = sum;
    }
}

int main() {
    // Matrix dimensions
    int k = 13, N = 3, M = 3, P = 3;

    // Get matrix dimensions from the user
    /*
    printf("Enter the number of matrices (k): ");
    scanf("%d", &k);

    printf("Enter the number of rows for matrices A (N): ");
    scanf("%d", &N);

    printf("Enter the number of columns for matrices A and rows for matrices B (M): ");
    scanf("%d", &M);

    printf("Enter the number of columns for matrices B (P): ");
    scanf("%d", &P);
    */

    // Host matrices
    int *h_A, *h_B, *h_C;
    // Device matrices
    int *d_A, *d_B, *d_C;

    // Allocate memory on the host
    h_A = (int *)malloc(k * N * M * sizeof(int));
    h_B = (int *)malloc(k * M * P * sizeof(int));
    h_C = (int *)malloc(k * N * P * sizeof(int));

    // Initialize matrices with random values
    for (int i = 0; i < k; ++i) {
        initializeMatrix(&h_A[i * N * M], N, M);
        initializeMatrix(&h_B[i * M * P], M, P);
    }

    // Allocate memory on the device
    cudaMalloc((void**)&d_A, k * N * M * sizeof(int));
    cudaMalloc((void**)&d_B, k * M * P * sizeof(int));
    cudaMalloc((void**)&d_C, k * N * P * sizeof(int));

    // Copy data from host to device
    cudaMemcpy(d_A, h_A, k * N * M * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, k * M * P * sizeof(int), cudaMemcpyHostToDevice);

    // Define grid and block dimensions
    dim3 threadsPerBlock(10, 10);
    dim3 blocksPerGrid((N + threadsPerBlock.x - 1) / threadsPerBlock.x,
                       (P + threadsPerBlock.y - 1) / threadsPerBlock.y,
                       k); // Add k as the third dimension for handling multiple matrices

     // Record start time
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    // Launch the CUDA kernel for element-wise matrix multiplication
    matrixMultiply<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N, M, P);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    printf("Time taken: %f milliseconds\n", milliseconds);

    // Copy result from device to host
    cudaMemcpy(h_C, d_C, k * N * P * sizeof(int), cudaMemcpyDeviceToHost);

    // Print the matrices and result for each pair
    for (int i = 0; i < k; ++i) {
        printf("\nMatrix A%d:\n", i + 1);
        printMatrix(&h_A[i * N * M], N, M);

        printf("\nMatrix B%d:\n", i + 1);
        printMatrix(&h_B[i * M * P], M, P);

        printf("\nResult Matrix C%d (Multiplication of A%d and B%d):\n", i + 1, i + 1, i + 1);
        printMatrix(&h_C[i * N * P], N, P);
    }

    // Free allocated memory
    free(h_A);
    free(h_B);
    free(h_C);
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing matrix.cu


In [ ]:
!nvcc -arch=sm_75 matrix.cu -o matrix

In [ ]:
!time ./matrix 400 2 2 2 2 > output1.txt


real	0m0.282s
user	0m0.015s
sys	0m0.215s


In [ ]:
#From Alamin code
%%writefile matrix.cu
#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void matrixMul(float *A, float *B, float *R, int M, int N, int P, int batchOffset) {
  int k = threadIdx.x + batchOffset;
  float *a = A + k * M * N;
  float *b = B + k * N * P;
  float *r = R + k * M * P;
for(int outer = 0; outer < 100; outer++) {
  for(int i = 0; i < M; i++) {
    for(int l = 0; l < P; l++) {
      r[i * P + l] = 0.0f; // explicitly set to 0
      for(int j = 0; j < N; j++) {
        r[i * P + l] += a[i * N + j] * b[j * P + l];
      }
    }
  }
}
}

// print the first matrix only
void printMatrix(float *A, int M, int N) {
  for(int i = 0; i < M; i++) {
    for(int j = 0; j < N; j++) {
      printf("%.0f ", A[i * N + j]);
    }
    cout<<endl;
  }
}

int main(int argc, char* argv[]) {
  int threads = atoi(argv[1]);
  int K = atoi(argv[2]);
  int M = atoi(argv[3]);
  int N = atoi(argv[4]);
  int P = atoi(argv[5]);

  //int K = 2, M = 2, N = 2, P = 2;

  int size_of_a = K * M * N;
  int size_of_b = K * N * P;
  int size_of_r = K * M * P;

  float *h_A = (float*)malloc(size_of_a * sizeof(float));
  float *h_B = (float*)malloc(size_of_b * sizeof(float));
  float *h_R = (float*)malloc(size_of_r * sizeof(float));

  for(int i = 0; i < size_of_a; i++) {
    h_A[i] = rand() % 10;
  }
  for(int i = 0; i < size_of_b; i++) {
    h_B[i] = rand() % 10;
  }
  float *d_A;
  cudaMalloc(&d_A, size_of_a * sizeof(float));
  cudaMemcpy(d_A, h_A, size_of_a * sizeof(float), cudaMemcpyHostToDevice);

  float *d_B;
  cudaMalloc(&d_B, size_of_b * sizeof(float));
  cudaMemcpy(d_B, h_B, size_of_b * sizeof(float), cudaMemcpyHostToDevice);

  float *d_R;
  cudaMalloc(&d_R, size_of_r * sizeof(float));
  cudaMemset(d_R, 0, size_of_r * sizeof(float));

  int remainingMatrices = K;
  int batchOffset = 0;

  while(remainingMatrices > 0) {
    int currentBatchSize = min(remainingMatrices, threads);
    matrixMul<<<1, currentBatchSize>>>(d_A, d_B, d_R, M, N, P, batchOffset);
    cudaDeviceSynchronize();
    remainingMatrices -= currentBatchSize;
    batchOffset += currentBatchSize;
  }

  cudaMemcpy(h_R, d_R, size_of_r * sizeof(float), cudaMemcpyDeviceToHost);

  cout<<"Matrix A[0]:"<<endl;
  printMatrix(h_A, M, N);
  cout<<"Matrix B[0]:"<<endl;
  printMatrix(h_B, N, P);
  cout<<"Matrix R[0]:"<<endl;
  printMatrix(h_R, M, P);
  cudaFree(d_A);
  cudaFree(d_B);
  cudaFree(d_R);
  free(h_A);
  free(h_B);
  free(h_R);
  return 0;
}

Writing matrix.cu


In [ ]:
!nvcc -arch=sm_75 matrix.cu -o matrix

In [ ]:
!time ./matrix 400 2 2 2 2 > output.txt


real	0m0.301s
user	0m0.021s
sys	0m0.215s


In [ ]:
#matrix multiplication cuda some changes
#index 9 r jonno

%%writefile matrix.cu
#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void matrixMul(float *A, float *B, float *R, int M, int N, int P, int batchOffset) {
  int k = threadIdx.x + batchOffset;
  float *a = A + k * M * N;
  float *b = B + k * N * P;
  float *r = R + k * M * P;
for(int outer = 0; outer < 100; outer++) {
  for(int i = 0; i < M; i++) {
    for(int l = 0; l < P; l++) {
      r[i * P + l] = 0.0f; // explicitly set to 0
      for(int j = 0; j < N; j++) {
        r[i * P + l] += a[i * N + j] * b[j * P + l];
      }
    }
  }
}
}

// print the first matrix only
void printMatrix(float *A, int M, int N) {
  for(int i = 0; i < M; i++) {
    for(int j = 0; j < N; j++) {
      printf("%.0f ", A[i * N + j]);
    }
    cout<<endl;
  }
}

int main(int argc, char* argv[]) {
  int threads = atoi(argv[1]);
  int K = atoi(argv[2]);
  int M = atoi(argv[3]);
  int N = atoi(argv[4]);
  int P = atoi(argv[5]);

  int idx = K-1;

  //int K = 2, M = 2, N = 2, P = 2;

  int size_of_a = K * M * N;
  int size_of_b = K * N * P;
  int size_of_r = K * M * P;

  float *h_A = (float*)malloc(size_of_a * sizeof(float));
  float *h_B = (float*)malloc(size_of_b * sizeof(float));
  float *h_R = (float*)malloc(size_of_r * sizeof(float));

  for(int i = 0; i < size_of_a; i++) {
    h_A[i] = rand() % 10;
  }
  for(int i = 0; i < size_of_b; i++) {
    h_B[i] = rand() % 10;
  }
  float *d_A;
  cudaMalloc(&d_A, size_of_a * sizeof(float));
  cudaMemcpy(d_A, h_A, size_of_a * sizeof(float), cudaMemcpyHostToDevice);

  float *d_B;
  cudaMalloc(&d_B, size_of_b * sizeof(float));
  cudaMemcpy(d_B, h_B, size_of_b * sizeof(float), cudaMemcpyHostToDevice);

  float *d_R;
  cudaMalloc(&d_R, size_of_r * sizeof(float));
  cudaMemset(d_R, 0, size_of_r * sizeof(float));

  int remainingMatrices = K;
  int batchOffset = 0;

  while(remainingMatrices > 0) {
    int currentBatchSize = min(remainingMatrices, threads);
    matrixMul<<<1, currentBatchSize>>>(d_A, d_B, d_R, M, N, P, batchOffset);
    cudaDeviceSynchronize();
    remainingMatrices -= currentBatchSize;
    batchOffset += currentBatchSize;
  }

  cudaMemcpy(h_R, d_R, size_of_r * sizeof(float), cudaMemcpyDeviceToHost);

cout<<"Matrix A[9]:"<<endl;
printMatrix(h_A + idx*M*N, M, N);

cout<<"Matrix B[9]:"<<endl;
printMatrix(h_B + idx*N*P, N, P);

cout<<"Matrix R[9]:"<<endl;
printMatrix(h_R + idx*M*P, M, P);

  cudaFree(d_A);
  cudaFree(d_B);
  cudaFree(d_R);
  free(h_A);
  free(h_B);
  free(h_R);
  return 0;
}



Overwriting matrix.cu


In [ ]:
!nvcc -arch=sm_75 matrix.cu -o matrix

In [ ]:
!time ./matrix 400 100 2 2 2 > output.txt


real	0m0.218s
user	0m0.011s
sys	0m0.203s


In [ ]:
#index 5 r jonno
%%writefile matrix.cu
#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void matrixMul(float *A, float *B, float *R, int M, int N, int P, int batchOffset) {
  int k = threadIdx.x + batchOffset;
  float *a = A + k * M * N;
  float *b = B + k * N * P;
  float *r = R + k * M * P;
for(int outer = 0; outer < 100; outer++) {
  for(int i = 0; i < M; i++) {
    for(int l = 0; l < P; l++) {
      r[i * P + l] = 0.0f; // explicitly set to 0
      for(int j = 0; j < N; j++) {
        r[i * P + l] += a[i * N + j] * b[j * P + l];
      }
    }
  }
}
}

// print the first matrix only
void printMatrix(float *A, int M, int N) {
  for(int i = 0; i < M; i++) {
    for(int j = 0; j < N; j++) {
      printf("%.0f ", A[i * N + j]);
    }
    cout<<endl;
  }
}

int main(int argc, char* argv[]) {
  int threads = atoi(argv[1]);
  int K = atoi(argv[2]);
  int M = atoi(argv[3]);
  int N = atoi(argv[4]);
  int P = atoi(argv[5]);

  //int K = 2, M = 2, N = 2, P = 2;
  //int index = 5;

  int size_of_a = K * M * N;
  int size_of_b = K * N * P;
  int size_of_r = K * M * P;

  float *h_A = (float*)malloc(size_of_a * sizeof(float));
  float *h_B = (float*)malloc(size_of_b * sizeof(float));
  float *h_R = (float*)malloc(size_of_r * sizeof(float));

  for(int i = 0; i < size_of_a; i++) {
    h_A[i] = rand() % 10;
  }
  for(int i = 0; i < size_of_b; i++) {
    h_B[i] = rand() % 10;
  }
  float *d_A;
  cudaMalloc(&d_A, size_of_a * sizeof(float));
  cudaMemcpy(d_A, h_A, size_of_a * sizeof(float), cudaMemcpyHostToDevice);

  float *d_B;
  cudaMalloc(&d_B, size_of_b * sizeof(float));
  cudaMemcpy(d_B, h_B, size_of_b * sizeof(float), cudaMemcpyHostToDevice);

  float *d_R;
  cudaMalloc(&d_R, size_of_r * sizeof(float));
  cudaMemset(d_R, 0, size_of_r * sizeof(float));

  int remainingMatrices = K;
  int batchOffset = 0;

  while(remainingMatrices > 0) {
    int currentBatchSize = min(remainingMatrices, threads);
    matrixMul<<<1, currentBatchSize>>>(d_A, d_B, d_R, M, N, P, batchOffset);
    cudaDeviceSynchronize();
    remainingMatrices -= currentBatchSize;
    batchOffset += currentBatchSize;
  }

  cudaMemcpy(h_R, d_R, size_of_r * sizeof(float), cudaMemcpyDeviceToHost);

  cout<<"Matrix A[5]:"<<endl;
  printMatrix(h_A, M, N);
  cout<<"Matrix B[5]:"<<endl;
  printMatrix(h_B, N, P);
  cout<<"Matrix R[5]:"<<endl;
  printMatrix(h_R, M, P);
  cudaFree(d_A);
  cudaFree(d_B);
  cudaFree(d_R);
  free(h_A);
  free(h_B);
  free(h_R);
  return 0;
}

Overwriting matrix.cu


In [ ]:
!nvcc -arch=sm_75 matrix.cu -o matrix

In [ ]:
!time ./matrix 400 2 2 2 2 > output.txt


real	0m0.258s
user	0m0.020s
sys	0m0.215s


//!nvcc -arch=sm_75 matrix.cu -o matrix
//!time ./matrix 400 2 2 2 2 > output.txt